# 03. Vectorization & Universal Functions: Beginner Guide

### 🌟 What is Vectorization & Universal Functions (ufuncs)?
In pure Python, applying math across a list requires a slow `for` loop. NumPy's **Vectorization** and **Universal Functions (ufuncs)** run mathematical computations in compiled C code across entire arrays at once, providing speedups of 50x to 100x.

This interactive guide loads and works directly with `data/raw_transactions.csv`, giving you real-world hands-on practice.

### 📚 Key Concepts Covered in this Notebook:
- **Arithmetic Vectorization**: Element-wise `+`, `-`, `*`, `/`, `**` executing in C without GIL contention.
- **Mathematical ufuncs**: Covers `np.sqrt()`, `np.exp()`, `np.log()`, `np.sin()`, and `np.abs()`.
- **Ufunc Reduction Methods**: Covers `.reduce()`, `.accumulate()`, and `.outer()`.


In [1]:
# Setup imports & dataset loading from raw_transactions.csv
import numpy as np
import pandas as pd
import sys
import time
import os

# Load raw transactions and extract aligned NumPy numeric arrays
csv_path = 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv'
raw_df = pd.read_csv(csv_path)
clean_raw = raw_df.dropna(subset=['transaction_amount', 'is_fraud', 'account_age_months']).reset_index(drop=True)
amounts = clean_raw['transaction_amount'].to_numpy(dtype=np.float64)
fraud_flags = clean_raw['is_fraud'].to_numpy(dtype=np.int8)
account_ages = clean_raw['account_age_months'].to_numpy(dtype=np.float32)

print(f"NumPy Version: {np.__version__}")
print(f"Loaded from {csv_path} ({len(amounts)} clean aligned rows):")
print(f"- amounts array: shape {amounts.shape}, dtype {amounts.dtype}")
print(f"- fraud_flags array: shape {fraud_flags.shape}, dtype {fraud_flags.dtype}")
print(f"- account_ages array: shape {account_ages.shape}, dtype {account_ages.dtype}")

NumPy Version: 1.26.4
Loaded from ../data/raw_transactions.csv (14262 clean aligned rows):
- amounts array: shape (14262,), dtype float64
- fraud_flags array: shape (14262,), dtype int8
- account_ages array: shape (14262,), dtype float32


### 🔹 Arithmetic Vectorization (`+`, `-`, `*`, `/`)
Applies fee multipliers and taxes element-wise to transaction amounts. NumPy runs in compiled C memory buffers, enabling mathematical operations across millions of numbers simultaneously in milliseconds. **Tip:** NumPy operations are optimized for homogeneous numeric data, offering massive speed improvements over standard Python loops.

**Syntax:** `amounts[:5] * 1.05`


In [2]:
print('Amounts with 5% Fee Applied (first 5):', (amounts[:5] * 1.05).round(2))

Amounts with 5% Fee Applied (first 5): [1277.15  341.24  143.49  130.42 1348.91]


### 🔹 Square Root: `np.sqrt()`
Computes square root of transaction amounts for variance scaling. NumPy runs in compiled C memory buffers, enabling mathematical operations across millions of numbers simultaneously in milliseconds. **Tip:** NumPy operations are optimized for homogeneous numeric data, offering massive speed improvements over standard Python loops.

**Syntax:** `np.sqrt(amounts[:5])`


In [3]:
print('Square Roots of Amounts:', np.sqrt(amounts[:5]).round(2))

Square Roots of Amounts: [34.88 18.03 11.69 11.14 35.84]


### 🔹 Exponential: `np.exp()`
Computes exponential scaling for risk logits. NumPy runs in compiled C memory buffers, enabling mathematical operations across millions of numbers simultaneously in milliseconds. **Tip:** NumPy operations are optimized for homogeneous numeric data, offering massive speed improvements over standard Python loops.

**Syntax:** `np.exp(account_ages[:5] / 12.0)`


In [4]:
print('Exponential Account Age Factors:', np.exp(account_ages[:5] / 12.0).round(2))

Exponential Account Age Factors: [  106.34 11308.76   289.07    64.5   2980.96]


### 🔹 Natural Logarithm: `np.log()`
Applies log-transformation $\ln(1 + x)$ to normalize skewed transaction amounts. NumPy runs in compiled C memory buffers, enabling mathematical operations across millions of numbers simultaneously in milliseconds. **Tip:** NumPy operations are optimized for homogeneous numeric data, offering massive speed improvements over standard Python loops.

**Syntax:** `np.log1p(amounts[:5])`


In [5]:
log_amounts = np.log1p(amounts[:5])
print('Log-Transformed Amounts (ln(1+x)):', log_amounts.round(3))

Log-Transformed Amounts (ln(1+x)): [7.104 5.787 4.925 4.83  7.159]


### 🔹 Trigonometric Functions: `np.sin()` & `np.cos()`
Computes cyclical trigonometric time encodings. NumPy runs in compiled C memory buffers, enabling mathematical operations across millions of numbers simultaneously in milliseconds. **Tip:** NumPy operations are optimized for homogeneous numeric data, offering massive speed improvements over standard Python loops.

**Syntax:** `np.sin(2 * np.pi * day_of_year / 365.25)`


In [6]:
cyclical_feature = np.sin(2 * np.pi * (amounts[:5] % 365) / 365)
print('Cyclical Encodings:', cyclical_feature.round(3))

Cyclical Encodings: [ 0.869 -0.636  0.71   0.843 -0.123]


### 🔹 Absolute Value: `np.abs()`
Computes absolute deviation from median transaction amount. NumPy runs in compiled C memory buffers, enabling mathematical operations across millions of numbers simultaneously in milliseconds. **Tip:** NumPy operations are optimized for homogeneous numeric data, offering massive speed improvements over standard Python loops.

**Syntax:** `np.abs(amounts[:5] - np.median(amounts))`


In [7]:
abs_devs = np.abs(amounts[:5] - np.median(amounts))
print('Absolute Deviations from Median:', abs_devs.round(2))

Absolute Deviations from Median: [216.43 674.9  863.24 875.68 284.79]


### 🔹 Cumulative Reductions with `ufunc.reduce()`
Calculates total revenue using `np.add.reduce`. NumPy runs in compiled C memory buffers, enabling mathematical operations across millions of numbers simultaneously in milliseconds. **Tip:** NumPy operations are optimized for homogeneous numeric data, offering massive speed improvements over standard Python loops.

**Syntax:** `np.add.reduce(amounts)`


In [8]:
total_revenue = np.add.reduce(amounts)
print(f'Total Transaction Volume (np.add.reduce): ${total_revenue:,.2f}')

Total Transaction Volume (np.add.reduce): $14,349,538.14


### 🔹 Running Totals with `ufunc.accumulate()`
Calculates running cumulative revenue stream across transactions. NumPy runs in compiled C memory buffers, enabling mathematical operations across millions of numbers simultaneously in milliseconds. **Tip:** NumPy operations are optimized for homogeneous numeric data, offering massive speed improvements over standard Python loops.

**Syntax:** `np.add.accumulate(amounts[:5])`


In [9]:
running_revenue = np.add.accumulate(amounts[:5])
print('Running Cumulative Revenue (first 5):', running_revenue.round(2))

Running Cumulative Revenue (first 5): [1216.33 1541.32 1677.98 1802.19 3086.87]


### 🔹 Outer Products with `ufunc.outer()`
Computes outer cross-product risk matrix between card fee tiers and account ages. NumPy runs in compiled C memory buffers, enabling mathematical operations across millions of numbers simultaneously in milliseconds. **Tip:** NumPy operations are optimized for homogeneous numeric data, offering massive speed improvements over standard Python loops.

**Syntax:** `np.multiply.outer(fee_rates, account_ages[:4])`


In [10]:
fees = np.array([0.015, 0.025, 0.035])
outer_fee_matrix = np.multiply.outer(fees, account_ages[:4])
print('Outer Fee Scaling Matrix (3 fees x 4 accounts):\n', outer_fee_matrix.round(2))

Outer Fee Scaling Matrix (3 fees x 4 accounts):
 [[0.84 1.68 1.02 0.75]
 [1.4  2.8  1.7  1.25]
 [1.96 3.92 2.38 1.75]]


## 💡 Real-World Practice & Scenarios
Practical scenarios and common data questions explained simply with real examples.


### 🔍 Scenario: Q1: Pairwise Transaction Distance Matrix via `np.subtract.outer`

**Approach:** Compute the complete pairwise absolute difference matrix for the first 5 transaction amounts without Python loops.
**Syntax:** `np.abs(np.subtract.outer(amounts[:5], amounts[:5]))`


In [11]:
pw_matrix = np.abs(np.subtract.outer(amounts[:5], amounts[:5]))
print('Pairwise Amount Differences Matrix:\n', pw_matrix.round(2))

Pairwise Amount Differences Matrix:
 [[   0.    891.34 1079.67 1092.12   68.35]
 [ 891.34    0.    188.33  200.78  959.69]
 [1079.67  188.33    0.     12.45 1148.02]
 [1092.12  200.78   12.45    0.   1160.47]
 [  68.35  959.69 1148.02 1160.47    0.  ]]
